In [ ]:
# ====================================================================================
# Abstract:
#        	Load Unsupervised
# Inputs:
#   Context:
#   Variables (inc global):
#
# Outputs:
#   Context:
#   Variables:
#
# Change History:
# Date			By					Reason
# 06-May-2023	Randle Knight		Initial Version
#  ====================================================================================

In [ ]:
## Execution timing
import time
from datetime import datetime
startTime = time.time()

In [ ]:
## Load h2o and prerequisites libraries
import h2o
import numpy as np
import pandas as pd
from h2o.estimators import H2OExtendedIsolationForestEstimator
import warnings
import json
import pyodbc # import oracledb

import config

In [ ]:
h2o.init(nthreads=-1, min_mem_size= config.min_memory, max_mem_size= config.max_memory, enable_assertions=False)
warnings.filterwarnings('ignore')
# Clear h2o memory
h2o.remove_all()

In [ ]:
# Check lims dsn connection to database before proceeding
def get_lims_connection():
    global odbcLimsConnection
    if config.dbType == "oracle":
        import oracledb
        oracledb.init_oracle_client(config.lib_dir)
        odbcLimsConnection = oracledb.connect(user=config.limsdb_username, 
                                              password=config.limsdb_password,
                                              dsn=config.limsdb_dsn)
        odbcLimsConnection.arraysize = 5000
        
    elif config.dbType == "postgres":
        import psycopg
        odbcLimsConnection = psycopg.connect('dbname=' + config.limsdb_database + ' user=' + config.limsdb_username + 
                                             ' password=' + config.limsdb_password + ' host=' + config.limsdb_hostname)
    
    else:
        import pyodbc
        odbcLimsConnection = pyodbc.connect('DRIVER=' + config.limsdb_driver + ';DATABASE=' + config.limsdb_database +
                                        ';UID=' + config.limsdb_username + ';PWD=' + config.limsdb_password +
                                        ';SERVER=' + config.limsdb_hostname + ';PORT=' + config.limsdb_port +
                                        ';Trusted_Connection=yes')
    return odbcLimsConnection


if __name__ == '__main__':
    try:  # GET THE CONNECTION OBJECT (ENGINE) FOR THE DATABASE
        get_lims_connection()
    except Exception as error:
        exit("Connection could not be made due to the following error: \n", error)


# Check mlar dsn connection to database before proceeding
def get_mlar_connection():
    global odbcMlarConnection
    if config.dbType == "oracle":
        import oracledb
        odbcMlarConnection = oracledb.connect(user=config.mlardb_username, password=config.mlardb_password,dsn=config.mlardb_dsn)
        odbcMlarConnection.cursor().arraysize=5000
        
    elif config.dbType == "postgres":
        import psycopg
        odbcMlarConnection = psycopg.connect('dbname=' + config.mlardb_database + ' user=' + config.mlardb_username + 
                                             ' password=' + config.mlardb_password + ' host=' + config.mlardb_hostname)        
    else:
        import pyodbc
        odbcMlarConnection = pyodbc.connect('DRIVER=' + config.mlardb_driver + ';DATABASE=' + config.mlardb_database +
                                        ';UID=' + config.mlardb_username + ';PWD=' + config.mlardb_password +
                                        ';SERVER=' + config.mlardb_hostname + ';PORT=' + config.mlardb_port +
                                        ';Trusted_Connection=yes')
    return odbcMlarConnection


if __name__ == '__main__':
    try:
        # GET THE CONNECTION OBJECT (ENGINE) FOR THE DATABASE
        get_mlar_connection()
    except Exception as error:
        exit("Connection could not be made due to the following error: \n", error)


In [ ]:
# Model location
#model_path = config.model_path

In [ ]:
# Start and Finish Logs - MOVE TO CONFIG 
# Daily anomaly check
#sql_command = "select CAST(TO_DATE(MIN(AUDIT_TIMESTAMP),'dd-Mon-YYYY') as date), cast(TO_DATE(ADD_DAYS(MIN(AUDIT_TIMESTAMP), 1),'dd-Mon-YYYY') as date) from SAMPLE_AUDIT_DECRYPT " # Oracle handling
#sql_command = "select MIN(AUDIT_TIMESTAMP),MIN(AUDIT_TIMESTAMP) + INTERVAL '1 day' from SAMPLE_AUDIT_DECRYPT" 
sql_command = "select MIN(AUDIT_TIMESTAMP),MAX(AUDIT_TIMESTAMP) from SAMPLE_AUDIT_DECRYPT" #Load everything
with odbcMlarConnection.cursor() as lims_cursor_fetch_one:
    lims_cursor_fetch_one.execute(sql_command)
    response = lims_cursor_fetch_one.fetchone()

startLog = response[0]
endLog = response[1]

In [ ]:
# Execution timing
endTime = time.time()
elapsedTime = endTime - startTime
now = datetime.now()
logEntry = "DateTime: " + str(now) + ", Start: " + str(startLog) + ", End: " + str(endLog) + ", Sample Audit, " \
          "Prerequisite Load End Time = " + str(elapsedTime)   
with open ("executionlog.txt", "a") as writer:
    writer.write('\n'+logEntry)
    writer.close()
    print(logEntry)

In [ ]:
# Query data and load into dataframe
# bulk union needs to applied for all audit tables
column_names = ['MLAR_TYPE', 'AUDIT_ACTION', 'AUDIT_TIMESTAMP', 'AUDIT_TYPE', 
                'AUDIT_COUNTER', 'TRANS_ORDER', 'TABLE_NAME', 'TABLE_KEY', 'REASON', 
                'RECORD_SIGNED', 'USER_NAME',  'USER_ROLE', 'SAMPLE_NUMBER', 'T_MLAR',
                'T_MLAR_SEVERITY', 'TRANS_STRING']
sql_command = "select 'sample_audit_log',action,audit_timestamp,audit_type,counter,trans_order,table_name,table_key," \
              "reason,record_signed,user_name,user_role,sample_number,t_mlar,t_mlar_severity,trans_string" \
              " from SAMPLE_AUDIT_DECRYPT where audit_timestamp " \
             "between '" + str(startLog) + "' and '" + str(endLog) + "'"

dataFrame = pd.read_sql(sql_command, odbcMlarConnection, columns=column_names)
dataFrame = dataFrame.astype({'trans_string': str}) #Added for pd.read_sql handling

In [ ]:
# Retrieve datatypes for training Dataframe
# "string" | "float" | "real" | "double" | "int" | "long" | "numeric" | "categorical" | "factor" | "enum" | "time"
sql_command = "select distinct FIELD_NAME, CASE WHEN FIELD_NAME = 'T_MLAR_SEVERITY' THEN 'categorical' WHEN DATA_TYPE " \
              "in ('Text','Long','List','MultiList','Boolean','File','DBFile','Interval','DateTime','Date') THEN " \
              "'string' WHEN DATA_TYPE in ('Integer','Number') THEN 'numeric' ELSE DATA_TYPE END AS DATATYPE from " \
              "FIELD_MASTER where TABLE_NAME in ('SAMPLE_AUDIT_LOG','SAMPLE','TEST','RESULT') order by FIELD_NAME"

with odbcLimsConnection.cursor() as lims_cursor_fetch_all:
    lims_cursor_fetch_all.execute(sql_command)
    queryFetch = lims_cursor_fetch_all.fetchall()
    
fieldDict = queryFetch

In [ ]:
#JSON data transformation
dataFrame['trans_string'] = dataFrame.trans_string.str.replace("\r\n", " ", regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace("\n", " ", regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace("\r", " ", regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r'" ', r"> ", regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r' "', r" <", regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r' <,"', r' <","' , regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r'> <"', r'" "' , regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r'<}', r'"}' , regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r'""[A-Z]', r'"[A-Z]' , regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r'\\', r' ' , regex=True)

In [ ]:
# Convert JSON field to JSON
def loads_json(x):
   try:
       return json.loads(x, strict=False) if pd.notna(x) else pd.Series() #
   except json.JSONDecodeError as e:
       print(f"Error decoding JSON: {e}")
       print(f"Problematic JSON string: {x}")
       return pd.Series()
    
dataFrame['trans_string'] = dataFrame['trans_string'].apply(loads_json)
jsonDataFrame = pd.json_normalize(dataFrame['trans_string'])

In [ ]:
# Convert column datatypes to dictionary
fieldDictDF = pd.DataFrame.from_records(fieldDict)
fieldDictDF.set_index(0, inplace=True)
fieldDict = fieldDictDF[1].to_dict()

In [ ]:
# Drop unnecessary or duplicated columns, replace NaN values and comma-replacement character
dataFrame = dataFrame.drop(columns=['trans_string'])
jsonDataFrame = jsonDataFrame.drop(columns=['SAMPLE_NUMBER']).replace(np.nan, "█").replace("▄", ",", regex=True) #.drop(columns=['sample_number'])
encodedDataFrame = pd.concat([dataFrame, jsonDataFrame], axis=1)


In [ ]:
# Handling for experiment_02 
filtered = True
if filtered == True:
    outputSubFolder = "/grid_filtered"
    standard_labware_fields = ['TEXT_ID','STATUS','OLD_STATUS','IN_CAL','RE_SAMPLE','ORIGINAL_SAMPLE','ALIQUOT','PARENT_SAMPLE','PARENT_ALIQUOT','LOGIN_DATE','LOGIN_BY','RECD_DATE','RECEIVED_BY','STARTED','DATE_REVIEWED','PREP_DATE','PREP_BY','PRODUCT_VERSION','PROJECT','BATCH_NAME','BATCH_TEMPLATE','STEP','SAMPLE_TYPE','TEMPLATE','STANDARD','LOT','CHANGED_ON','PARTIAL_SPEC','PRIMARY_IN_SPEC','PROTOCOL_LEVEL','RELEASED','RELEASED_BY','RELEASED_ON','INVESTIGATION','INVESTIGATED','MODIFIED_RESULTS','COMPOSITE','PARENT_COMPOSITE','HAS_FLAGS','IMPORTANT_MESSAGE','SIGNED','TEST_NUMBER','CHARGE_ENTRY','VERSION','REPLICATE_COUNT','DATE_RECEIVED','DATE_STARTED','PREP','DATE_COMPLETED','REPLICATE_TEST','ORDER_NUMBER','GROUP_NAME','RESOLVE_REQD','STAGE','BATCH','IN_CONTROL','VARIATION','RE_TESTED','TEST_LIST','ALIQUOTED_TO','ON_WORKSHEET','ANALYSIS_COUNT','BATCH_ORIGINAL_TEST','BATCH_PARENT_TEST','CHILD_OUT_SPEC','CROSS_SAMPLE','ORIGINAL_TEST','PARENT_TEST','RESULT_NUMBER','NAME','RESULT_TYPE','MINIMUM','MAXIMUM','ALLOW_OUT','FORMATTED_ENTRY','NUMERIC_ENTRY','PLACES','ENTERED_ON','ENTERED_BY','USES_CODES','AUTO_CALC','LIST_KEY','ALLOW_CANCEL','REPORTABLE','OPTIONAL','CODE_ENTERED','ROUND','REVIEWER','STD_REAG_SAMPLE','DISPLAYED','ENTRY_QUALIFIER','INSTRUMENT','ANALYSIS','USES_INSTRUMENT','HAS_ATTRIBUTES','FACTOR_OPERATOR','ENTRY_TYPE','ALIAS_NAME','CONTROL_1','CONTROL_2','REPORTED_NAME']
    audit_fields = ['action','audit_timestamp','audit_type','counter','trans_order','table_name','table_key','reason','record_signed','user_name','user_role','sample_number','t_mlar','t_mlar_severity']
    filter_fields = standard_labware_fields + audit_fields 
    encodedDataFrame = encodedDataFrame.filter(items=filter_fields)
else:
    outputSubFolder = "/grid_unfiltered"
    pass
unsupervisedFeatures = list(encodedDataFrame.drop(columns=['t_mlar']).columns.values) 

In [ ]:
# Define datatypes and convert to H2oFrame
fieldDictUnsupervised = dict(filter(lambda item: item[0] in supervisedFeatures, fieldDict.items()))
unsupervisedH2oFrame = h2o.H2OFrame(python_obj=encodedDataFrame, column_types=fieldDictSupervised)
#unsupervisedH2oFrame["t_mlar_severity"] = unsupervisedH2oFrame["t_mlar_severity"].asfactor()

In [ ]:
# Execution timing
endTime = time.time()
elapsedTime = endTime - startTime
now = datetime.now()
logEntry = "DateTime: " + str(now) + ", Start: " + str(startLog) + ", End: " + str(endLog) + ", Sample Audit," \
    "json convert Time = " + str(elapsedTime)
with open ("executionlog.txt", "a") as writer:
    writer.write('\n'+logEntry)
    writer.close()
    print(logEntry)

In [ ]:
# RULES ENGINE
#'Retrieve user-defined detection settings'
# General settings
#column_names = ['filter_logged_samps', 'filter_cancelled_samps', 'auto_assign_sev', 'auto_severity']
#sql_command = "select filter_logged_samps, filter_cancelled_samps, auto_assign_sev, auto_severity " \
#             "from T_MLAR_SETTING"
#dataFrame = pd.read_sql(sql_command, odbcLimsConnection, columns=column_names)

In [ ]:
# Granular customisable settings
#column_names = ['Name', 'auto_assign_sev', 'TABLE1', 'FIELD1', 'OPERATION', 'TABLE2', 'FIELD2', 'SEVERITY']
#sql_command = "select tas.NAME, tas.AUTO_ASSIGN_SEV, tasp.TABLE1, tasp.FIELD1, tasp.OPERATION, tasp.TABLE2, " \
#              "tasp.FIELD2, tasp.SEVERITY from T_MLAR_SETTING tas " \
#              "inner join T_MLAR_SETTING_PARAM tasp on tas.NAME = tasp.T_MLAR_SETTING "
#dataFrame = pd.read_sql(sql_command, odbcLimsConnection, columns=column_names)

In [ ]:
# Execution timing
endTime = time.time()
elapsedTime = endTime - startTime
now = datetime.now()
logEntry = "DateTime: " + str(now) + ", Start: " + str(startLog) + ", End: " + str(endLog) + ", Sample Audit," \
    "Apply Detection Settings Time = " + str(elapsedTime)
with open ("executionlog.txt", "a") as writer:
    writer.write('\n'+logEntry)
    writer.close()
    print(logEntry)

In [ ]:
# Execution timing
endTime = time.time()
elapsedTime = endTime - startTime
now = datetime.now()
logEntry = "DateTime: " + str(now) + ", Start: " + str(startLog) + ", End: " + str(endLog) + ", Sample Audit," \
    "Model Load Time = " + str(elapsedTime)
with open ("executionlog.txt", "a") as writer:
    writer.write('\n'+logEntry)
    writer.close()
    print(logEntry)

In [ ]:
gridDirectory = "tmp/grid_" + config.limsdb_database + outputSubFolder
supervisedModelPerformance = supervisedModel.model_performance(supervisedH2oFrameTest)


In [ ]:
supervisedModelPerformance

In [ ]:
#outputPerformance = pd.DataFrame(supervisedModelPerformance, index=[0]) 
#outputPerformance = outputPerformance.replace("nan", '', regex=True)
#outputLocation = gridDirectory + "/model_performance_locus_on_jag.csv"#
outputPerformance.to_csv(outputLocation, sep=',', encoding='utf-8')

In [ ]:
#supervisedModelPerformance = str(supervisedModelPerformance)

In [ ]:
# Execution timing
endTime = time.time()
elapsedTime = endTime - startTime
now = datetime.now()
logEntry = "DateTime: " + str(now) + ", Start: " + str(startLog) + ", End: " + str(endLog) + ", Sample Audit," \
    "Completion Time = " + str(elapsedTime)
with open ("executionlog.txt", "a") as writer:
    writer.write('\n'+logEntry)
    writer.close()
    print(logEntry)

In [ ]:
## Clear h2o memory
h2o.remove_all()
h2o.cluster().shutdown()